# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os
import json
import chromadb
import requests

from openai import OpenAI
from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from dotenv import load_dotenv
from chromadb.utils import embedding_functions
from pathlib import Path
from pydantic import BaseModel, Field

In [ ]:
_env_path = next(
    (p for p in [Path('.env'), Path('project/starter/.env')] if p.exists()),
    Path('.env')
)
load_dotenv(dotenv_path=_env_path, override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

openai_client = OpenAI()
print(f"Loaded OPENAI_API_KEY: {'set' if OPENAI_API_KEY else 'NOT SET'}")
print(f"Loaded TAVILY_API_KEY: {'set' if TAVILY_API_KEY else 'NOT SET'}")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
@tool
def retrieve_game(query: str) -> dict:
    """
    Semantic search: Finds the most relevant results in the vector DB.

    Args:
        query: a question about the game industry.

    Returns a dict with:
    - documents: list of document text strings (pass these to evaluate_retrieval)
    - details: list of detailed results, each containing Platform, Name,
      YearOfRelease, Description, and distance score
    """
    results = collection.query(
        query_texts=[query],
        n_results=3,
        include=["documents", "metadatas", "distances"]
    )

    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    details = []
    for doc, meta, dist in zip(docs, metas, distances):
        details.append({
            "text": doc,
            "Platform": meta.get("Platform"),
            "Name": meta.get("Name"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Description": meta.get("Description"),
            "distance": round(dist, 4),
        })

    return {
        "documents": docs,
        "details": details,
    }

In [ ]:
# Instantiate ChromaDB Client
chroma_client = chromadb.PersistentClient(path="chromadb")

In [ ]:
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv('CHROMA_OPENAI_API_KEY'),
    api_base=os.getenv('OPENAI_API_BASE')
)

In [ ]:
# Get or create collection (avoids error if collection already exists)
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn)

    
print(f"Collection '{collection.name}' is ready — {collection.count()} documents currently indexed.")

In [ ]:
# Demonstrate semantic search works correctly
print("=== Semantic Search Demo ===\n")
demo_results = collection.query(
    query_texts=["Which games are about realistic racing?"],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

docs = demo_results.get("documents", [[]])[0]
metas = demo_results.get("metadatas", [[]])[0]
distances = demo_results.get("distances", [[]])[0]

print(f"Query: 'Which games are about realistic racing?'\n")
for i, (doc, meta, dist) in enumerate(zip(docs, metas, distances), 1):
    print(f"Result {i} (distance: {dist:.4f}):")
    print(f"  Name: {meta.get('Name')} ({meta.get('YearOfRelease')})")
    print(f"  Platform: {meta.get('Platform')}")
    print(f"  Text: {doc}\n")

#### Evaluate Retrieval Tool

In [ ]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

In [ ]:
from pydantic import BaseModel, Field


class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the retrieved documents are useful to answer the question")
    description: str = Field(description="Explanation of why the documents are or are not sufficient")


@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.

    Args:
        question: original question from user
        retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    Returns a dict with:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    docs_text = "\n\n".join(str(doc) for doc in retrieved_docs)

    prompt = f"""Your task is to evaluate if the documents are enough to respond the query.
Give a detailed explanation, so it's possible to take an action to accept it or not.

User question:
{question}

Retrieved documents:
{docs_text}

Decide:
1. Are these documents useful enough to answer the question?
2. Explain why or why not.
3. Mention if important information is missing, incomplete, or irrelevant."""

    response = openai_client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format=EvaluationReport,
    )

    report = response.choices[0].message.parsed

    return {
        "useful": report.useful,
        "description": report.description,
    }

#### Game Web Search Tool

In [ ]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

In [ ]:
@tool
def game_web_search(question: str) -> dict:
    """
    Searches the web for information about video games using Tavily.
    Use this ONLY when the internal database retrieval was evaluated as not useful.

    Args:
        question: a question about the game industry.

    Returns a dict with:
    - answer: summarized answer from the web search
    - results: list of top search results with title, url, and content snippet
    - urls: list of source URLs for citation
    """
    url = "https://api.tavily.com/search"
    payload = {
        "api_key": TAVILY_API_KEY,
        "query": question,
        "include_answer": True,
        "include_raw_content": False,
        "include_images": False,
        "max_results": 3,
    }

    response = requests.post(url, json=payload, timeout=15)
    response.raise_for_status()
    data = response.json()

    top_results = data.get("results", [])[:3]

    return {
        "answer": data.get("answer", "No summary available."),
        "results": [
            {
                "title": r.get("title"),
                "url": r.get("url"),
                "content": r.get("content", "")[:300],
            }
            for r in top_results
        ],
        "urls": [r.get("url") for r in top_results],
    }

### Agent

In [ ]:
SYSTEM_INSTRUCTIONS = """You are UdaPlay, an expert AI research agent for the video game industry.

When answering any question, you MUST follow this EXACT workflow in order:

STEP 1 — Internal Retrieval: Always call retrieve_game first with the user's question to search the internal vector database.

STEP 2 — Evaluate Retrieval: Always call evaluate_retrieval next, passing the user's original question and the list from the 'documents' field of the retrieve_game result.

STEP 3 — Conditional Response:
- If evaluate_retrieval returns useful=True: Answer directly using the retrieved documents. Cite the game name, platform, and year.
- If evaluate_retrieval returns useful=False: Call game_web_search with the user's question, then answer using the web results. Include the source URLs in your final answer.

You MUST always call all three steps in order: retrieve_game → evaluate_retrieval → (answer OR game_web_search).
Never skip any step. Always cite your sources in the final answer."""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=SYSTEM_INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)

print("UdaPlay Agent initialized successfully.")
print(f"Tools: {[t.name for t in agent.tools]}")

In [ ]:
def display_agent_run(question: str, run):
    """Print the full reasoning trace and final answer from an agent run."""
    print(f"\n{'='*65}")
    print(f"QUERY: {question}")
    print('='*65)

    final_state = run.get_final_state()
    messages = final_state.get("messages", [])

    tools_used = []
    final_answer = ""

    for msg in messages:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                tool_name = tc.function.name
                tools_used.append(tool_name)
                print(f"\n[TOOL CALLED] {tool_name}")
                try:
                    args = json.loads(tc.function.arguments)
                    for k, v in args.items():
                        v_str = str(v)
                        print(f"  {k}: {v_str[:150]}{'...' if len(v_str) > 150 else ''}")
                except Exception:
                    pass
        elif hasattr(msg, 'role') and msg.role == 'tool':
            print(f"\n[TOOL RESULT] {msg.name}")
            try:
                content = json.loads(msg.content)
            except Exception:
                content = msg.content
            content_str = str(content)
            print(f"  {content_str[:400]}{'...' if len(content_str) > 400 else ''}")
        elif hasattr(msg, 'role') and msg.role == 'assistant' and msg.content:
            final_answer = msg.content

    print(f"\n[WORKFLOW] Tools used in order: {' → '.join(tools_used) if tools_used else 'None'}")
    print(f"\n[FINAL ANSWER]\n{final_answer}")
    print('='*65)

In [ ]:
# Invoke your agent with all 3 required queries:
# - When Pokémon Gold and Silver was released?        → found in DB → answer from DB
# - Which one was the first 3D platformer Mario game? → found in DB → answer from DB
# - Was Mortal Kombat X released for Playstation 5?  → NOT in DB  → web fallback + URLs

In [ ]:
queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
]

for question in queries:
    agent.reset_session()
    run = agent.invoke(question)
    display_agent_run(question, run)

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes